# Partie A — Web Scraping
**Source :** https://www.moteur.ma/fr/voiture/achat-voiture-occasion  
**Objectif :** Extraire les annonces de voitures d'occasion (marque, modèle, année, km, prix, etc.)

**Stratégie :**
1. Parcourir les pages de **listing** → collecter tous les liens vers les pages détail
2. Visiter chaque **page détail** → extraire les features
3. Sauvegarder en CSV

In [44]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import random

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

BASE_URL = "https://www.moteur.ma/fr/voiture/achat-voiture-occasion/"


In [45]:

"""
response = requests.get(BASE_URL, headers=HEADERS)
soup = BeautifulSoup(response.content, 'html.parser')
links = soup.find_all('a', href=True)

print(f"Nombre de liens trouvés : {len(links)}")
"""

'\nresponse = requests.get(BASE_URL, headers=HEADERS)\nsoup = BeautifulSoup(response.content, \'html.parser\')\nlinks = soup.find_all(\'a\', href=True)\n\nprint(f"Nombre de liens trouvés : {len(links)}")\n'

### 1.4 Extraction de tous les liens des pages de listing (pages d'acceuil + pages suivantes)

In [46]:

# --- Étape 1a : trouver le nombre total de pages ---
max_page = 300
"""
print(f"Nombre total de pages : {max_page}")
# Étape 1b : construire toutes les URLs de listing 
page_urls =  [
    f"https://www.moteur.ma/fr/voiture/achat-voiture-occasion?page={i}"
    for i in range(2, max_page + 1)
]
"""
# --- Étape 1c : fonction de scraping d'une page ---
def scrape_listing_page(url):
    """Scrape une page de listing et retourne les liens d'annonces de voitures d'occasion."""
    
    resp = requests.get(url, headers=HEADERS, timeout=15)
    s = BeautifulSoup(resp.content, 'html.parser')
    return [
        link['href'] for link in s.find_all('a', href=True)
        if re.match(r'^https://www.moteur.ma/fr/voiture/achat-voiture-occasion/detail-annonce/.+', link['href'])
    ]
'''
# --- Étape 1d : scraping concurrent (8 workers) ---
voiture_links = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(scrape_listing_page, url): url for url in page_urls}
    for future in as_completed(futures):
        links = future.result()
        voiture_links.extend(links)
        print(f"Page traitée — {len(links)} liens trouvés ({len(voiture_links)} total)")

voiture_links = list(set(voiture_links))  # dédoublonnage
print(f"\nNombre total de liens d'annonces : {len(voiture_links)}")
'''

'\n# --- Étape 1d : scraping concurrent (8 workers) ---\nvoiture_links = []\nwith ThreadPoolExecutor(max_workers=8) as executor:\n    futures = {executor.submit(scrape_listing_page, url): url for url in page_urls}\n    for future in as_completed(futures):\n        links = future.result()\n        voiture_links.extend(links)\n        print(f"Page traitée — {len(links)} liens trouvés ({len(voiture_links)} total)")\n\nvoiture_links = list(set(voiture_links))  # dédoublonnage\nprint(f"\nNombre total de liens d\'annonces : {len(voiture_links)}")\n'

In [47]:
# creation d'un dataframe pandas pour stocker les liens des annonces de voitures d'occasion
#df_links = pd.DataFrame(voiture_links, columns=['Lien'])
#df_links.to_csv('../data/liens_annonces_voitures.csv', index=True, index_label='index')

### Extraction des informations de chaque vehicule à partir de sa page détail

In [48]:

voiture_links = pd.read_csv('../data/liens_annonces_voitures.csv')['Lien'].tolist()
def extract_characteristics(soup):
    characteristics = {}
    info_section = soup.find('h3', class_='card-title mb-3 font-weight-semibold', string='Informations Véhicule')
    if info_section:
        info_table = info_section.find_next('table', class_='table table-bordered')
        if info_table:
            for row in info_table.find_all('tr'):
                cols = row.find_all('td')
                for i in [0, 2]:
                    if len(cols) >= i + 2:          # garde contre les lignes incomplètes
                        key = cols[i].get_text(strip=True).rstrip(':')
                        value = cols[i+1].get_text(strip=True) or None
                        if key:
                            characteristics[key] = value
    return characteristics

"""
# --- Test sur la première annonce ---
voiture_link = voiture_links[0]
response = requests.get(voiture_link, headers=HEADERS)
soup = BeautifulSoup(response.content, 'html.parser')
print(extract_characteristics(soup))
"""

"\n# --- Test sur la première annonce ---\nvoiture_link = voiture_links[0]\nresponse = requests.get(voiture_link, headers=HEADERS)\nsoup = BeautifulSoup(response.content, 'html.parser')\nprint(extract_characteristics(soup))\n"

In [49]:

def extract_price_and_city(soup):
    price_div = soup.find('div', class_='ad-hero-price-col')
    prix_raw = price_div.get_text(strip=True) if price_div else None
  
    try:
        prix = int(prix_raw.replace(',', '').replace('\xa0', '').replace('MAD', '').strip()) if prix_raw else None
    except (ValueError, TypeError):
        prix = None
    ville = None
    for span in soup.find_all('span', class_='ad-detail-meta__item'):
        if span.find('i', class_='fa-map-marker'):
            ville = span.get_text(strip=True)
            break

    return {'prix': prix, 'ville': ville}

"""
# --- Test sur la première annonce ---
response = requests.get(voiture_links[0], headers=HEADERS)
soup = BeautifulSoup(response.content, 'html.parser')
price_city = extract_price_and_city(soup)
print(f"Prix : {price_city['prix']} MAD")
print(f"Ville : {price_city['ville']}")
"""

'\n# --- Test sur la première annonce ---\nresponse = requests.get(voiture_links[0], headers=HEADERS)\nsoup = BeautifulSoup(response.content, \'html.parser\')\nprice_city = extract_price_and_city(soup)\nprint(f"Prix : {price_city[\'prix\']} MAD")\nprint(f"Ville : {price_city[\'ville\']}")\n'

In [50]:
OPTIONS_LIST = [
    "État du véhicule",
    "Airbags",
    "Navigation GPS",
    "Ordinateur de bord",
    "Limiteur de vitesse",
    "Climatisation",
    "Intérieur cuir",
    "Radar de recul",
]

def extract_options(soup):
    """Retourne un dict {option: 1/0} pour chaque option de OPTIONS_LIST."""
    present = set()

    option_h4 = soup.find('h4', class_='mb-4', string='options')
    if option_h4:
        options_row = option_h4.find_next_sibling('div')
        if options_row:
            present = {
                div.get_text(strip=True)
                for div in options_row.find_all('div', class_='d-flex align-items-center')
            }

    return {opt: int(opt in present) for opt in OPTIONS_LIST}

"""
# --- Test ---
response = requests.get(voiture_links[0], headers=HEADERS)
soup = BeautifulSoup(response.content, 'html.parser')
options = extract_options(soup)
print(options)
"""

"\n# --- Test ---\nresponse = requests.get(voiture_links[0], headers=HEADERS)\nsoup = BeautifulSoup(response.content, 'html.parser')\noptions = extract_options(soup)\nprint(options)\n"

### Extraction des options , prix , ville et autres pour toutes les voitures et sauvegarde dans un dataframe

In [51]:
voiture_links = pd.read_csv('../data/liens_annonces_voitures.csv')['Lien'].tolist()

def save_checkpoint():
    pd.DataFrame(voiture_data).to_csv(SAVE_PATH, index=False)


# --- Resume : charger les lignes deja traitees ---

SAVE_PATH = '../data/raw_moteur.csv'
if os.path.exists(SAVE_PATH) and os.path.getsize(SAVE_PATH) > 0:
    df_existing = pd.read_csv(SAVE_PATH)
    done_indices = set(df_existing['Lien_index'].dropna().astype(int))
    voiture_data = df_existing.to_dict('records')
    print(f"Resume : {len(done_indices)} annonces deja traitees")
else:
    done_indices = set()
    voiture_data = []



for idx, link in enumerate(voiture_links[745    :]):
    if idx in done_indices:
        continue
    try:
        time.sleep(random.uniform(1.01, 4.0))
        resp = requests.get(link, headers=HEADERS, timeout=15)
        s = BeautifulSoup(resp.content, 'html.parser')

        characteristics = extract_characteristics(s)
        price_city = extract_price_and_city(s)
        options = extract_options(s)

        voiture_data.append({
            'Lien_index': idx,
            **characteristics,
            **price_city,
            **options
        })

        if (idx + 1) % 50 == 0:
            save_checkpoint()
            print(f"{idx + 1}/{len(voiture_links)} traites — checkpoint sauvegarde")


    except Exception as e:
        print(f"Erreur index {idx} : {e}")
        save_checkpoint()  # sauvegarde immediate en cas d'erreur
        time.sleep(random.uniform(1.01, 4.0))
        continue
        


# --- Sauvegarde finale ---
save_checkpoint()
print(f"Termine : {len(voiture_data)} voitures sauvegardees dans {SAVE_PATH}")


Resume : 742 annonces deja traitees
750/8945 traites — checkpoint sauvegarde
Erreur index 755 : ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Erreur index 756 : HTTPSConnectionPool(host='www.moteur.ma', port=443): Read timed out.
800/8945 traites — checkpoint sauvegarde
Erreur index 803 : ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Erreur index 805 : ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
850/8945 traites — checkpoint sauvegarde
Erreur index 873 : ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
900/8945 traites — checkpoint sauvegarde
950/8945 traites — checkpoint sauvegarde
1000/8945 traites — checkpoint sauvegarde
1050/8945 traites — checkpoint sauvegarde
1100/89